In [14]:
pip install --upgrade --force-reinstall torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu128

^C
Note: you may need to restart the kernel to use updated packages.
Looking in indexes: https://download.pytorch.org/whl/cu128
  Using cached typing_extensions-4.16.0-py3-none-any.whl.metadata (3.3 kB)
  Using cached sympy-1.14.0-py3-none-any.whl.metadata (12 kB)



[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip
ERROR: Operation cancelled by user


#### PyTorch Fundamentals

Core data structure of PyTorch is a tensor. A multidimensional array with a shape and a data type. Kind of like an array, but with support for GPU and support auto-differentiation.

Creation of a tensor

In [1]:
import torch

# let's create a 2 x 3 array (tensor)
X = torch.tensor([[1, 2, 3], [4, 5, 6]])
X

tensor([[1, 2, 3],
        [4, 5, 6]])

Info

In [2]:
X.shape

torch.Size([2, 3])

In [3]:
X.dtype

torch.int64

Indexing

In [4]:
X[0, 1]

tensor(2)

In [5]:
X[:, 1]

tensor([2, 5])

#### Hardware Acceleration

In [7]:
if torch.cuda.is_available():
    device = "cude"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"

device

'cpu'

Now let's create a tensor on that GPU

In [8]:
M = torch.tensor([[1., 2., 3.], [4., 5., 6.]])
M = M.to(device)

In [9]:
M.device

device(type='cpu')

In [10]:
M = torch.tensor([[1., 2., 3.], [4., 5., 6.]], device=device)

Run something on the gpu

In [11]:
R = M @ M.T
R

tensor([[14., 32.],
        [32., 77.]])

In [15]:
M = torch.rand((1000, 1000))  # on the CPU
%timeit M @ M.T

6.13 ms ± 75.2 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [16]:
M = torch.rand((1000, 1000), device="cuda")  # on the GPU
%timeit M @ M.T

AssertionError: Torch not compiled with CUDA enabled

#### Autograd

PyTorch comes with an efficient implementation of reverse-mode auto-differentiation, called `autograd`, which stands for `automatic gradients`.

Consider a simple function, f(x) = x^2. Derivitive of this is f'(x) = 2x so eval on f(x) and f'(x) should be 25 and 10.

In [21]:
x = torch.tensor(5.0, requires_grad=True)
f = x ** 2
f

tensor(25., grad_fn=<PowBackward0>)

In [22]:
f.backward()
x.grad

tensor(10.)

After computing the gradients, you generally want to perform a gradien descent step by subtracting a fraction of the gradients from the model variables.

In [23]:
learning_rate = 0.1
with torch.no_grad():
    x -= learning_rate * x.grad

or use `detach()` which is detached from the computational graph.

In [24]:
x_detached = x.detach()
x_detached -= learning_rate * x.grad

Before repeating the process (forward pass + backward pass + gradient descent step), it is essential to zero out the gradients of every model params.

In [25]:
x.grad.zero_()

tensor(0.)

Putting it all together, this how the training loop looks like:

In [26]:
learning_rate = 0.1
x = torch.tensor(5.0, requires_grad=True)
for iteration in range(100):
    f = x ** 2 # forward pass
    f.backward() # backward pass
    with torch.no_grad():
        x -= learning_rate * x.grad # gradient descent step

    x.grad.zero_()

#### Implementing Linear Regression

In [27]:
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split

housing = fetch_california_housing()
X_train_full, X_test, y_train_full, y_test = train_test_split(
    housing.data, housing.target, random_state=42)
X_train, X_valid, y_train, y_valid = train_test_split(
    X_train_full, y_train_full, random_state=42)

KeyboardInterrupt: 

Let's cast our dataset to FloatTensor

In [ ]:
X_train = torch.FloatTensor(X_train)
X_valid = torch.FloatTensor(X_valid)
X_test = torch.FloatTensor(X_test)
means = X_train.mean(dim=0, keepdims=True)
stds = X_train.std(dim=0, keepdims=True)
X_train = (X_train - means) / stds
X_valid = (X_valid - means) / stds
X_test = (X_test - means) / stds

And convert our targets as well

In [ ]:
y_train = torch.FloatTensor(y_train).reshape(-1, 1)
y_valid = torch.FloatTensor(y_valid).reshape(-1, 1)
y_test = torch.FloatTensor(y_test).reshape(-1, 1)

Let's create the params for our model

In [ ]:
torch.manual_seed(42)
n_features = X_train.shape[1] # there are 8 input features
w = torch.rand((n_features, 1), requires_grad=True) # weights
b = torch.tensor(0., requires_grad=True) # bias

Next, let's train our model.

In [ ]:
learning_rate = 0.4
n_epochs = 20
for epoch in range(n_epochs):
    y_pred = X_train @ w + b
    loss = ((y_pred - y_train) ** 2).mean()
    loss.backward()
    with torch.no_grad():
        w -= learning_rate * w.grad
        b -= learning_rate * b.grad
        w.grad.zero_()
        b.grad.zero_()
    print(f"Epoch {epoch + 1}/{n_epochs}, Loss: {loss.item()}")

Now we can predict

In [28]:
X_new = X_test[:3] # pretend these are new instances
with torch.no_grad():
    y_pred = X_new @ w + b # use the trained params to make predictions
y_pred

NameError: name 'X_test' is not defined

Now let's rewrite using higher-level API.